In [1]:
#Import Library
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns

In [2]:
#Loading the dataset
df = pd.read_csv("../dataset/cfpb_data.csv", encoding="latin1")
df.head()

,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Consumer consent provided?,Submitted via,Date sent to company,Company response to consumer,Timely response?,Consumer disputed?,Complaint ID
0,2020-07-06,"Credit reporting, credit repair services, or o...",Credit reporting,Incorrect information on your report,Information belongs to someone else,NaN,Company has responded to the consumer and the ...,Experian Information Solutions Inc.,FL,346XX,NaN,Other,Web,2020-07-06,Closed with explanation,Yes,NaN,3730948
1,2019-12-26,Credit card or prepaid card,General-purpose credit card or charge card,"Advertising and marketing, including promotion...",Confusing or misleading advertising about the ...,NaN,NaN,CAPITAL ONE FINANCIAL CORPORATION,CA,94025,NaN,Consent not provided,Web,2019-12-26,Closed with explanation,Yes,NaN,3477549
2,2020-05-08,"Credit reporting, credit repair services, or o...",Credit reporting,Incorrect information on your report,Information belongs to someone else,These are not my accounts.,Company has responded to the consumer and the ...,Experian Information Solutions Inc.,NV,89030,NaN,Consent provided,Web,2020-05-08,Closed with explanation,Yes,NaN,3642453
3,2024-01-05,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Information belongs to someone else,Kindly address this issue on my credit report....,Company has responded to the consumer and the ...,Experian Information Solutions Inc.,IL,60502,NaN,Consent provided,Web,2024-01-05,Closed with non-monetary relief,Yes,NaN,8113747
4,2024-01-21,Credit reporting or other personal consumer re...,Credit reporting,Improper use of your report,Credit inquiries on your report that you don't...,NaN,Company has responded to the consumer and the ...,Experian Information Solutions Inc.,NC,27401,Servicemember,Consent not provided,Web,2024-01-21,Closed with explanation,Yes,NaN,8191825


In [3]:
df.shape

(14889549, 18)

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 14889549 entries, 0 to 14889548
Data columns (total 18 columns):
 #   Column                        Dtype
---  ------                        -----
 0   Date received                 str  
 1   Product                       str  
 2   Sub-product                   str  
 3   Issue                         str  
 4   Sub-issue                     str  
 5   Consumer complaint narrative  str  
 6   Company public response       str  
 7   Company                       str  
 8   State                         str  
 9   ZIP code                      str  
 10  Tags                          str  
 11  Consumer consent provided?    str  
 12  Submitted via                 str  
 13  Date sent to company          str  
 14  Company response to consumer  str  
 15  Timely response?              str  
 16  Consumer disputed?            str  
 17  Complaint ID                  int64
dtypes: int64(1), str(17)
memory usage: 2.0 GB


### Data Cleaning & Preparation

In [5]:
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_").str.replace("-", "_").str.replace(r"[^\w]", "",regex=True)
df.columns

Index(['date_received', 'product', 'sub_product', 'issue', 'sub_issue',
       'consumer_complaint_narrative', 'company_public_response', 'company',
       'state', 'zip_code', 'tags', 'consumer_consent_provided',
       'submitted_via', 'date_sent_to_company', 'company_response_to_consumer',
       'timely_response', 'consumer_disputed', 'complaint_id'],
      dtype='str')

In [6]:
#Checking for missing values
df.isnull().sum()

date_received                          0
product                                0
sub_product                       235276
issue                                  6
sub_issue                         903347
consumer_complaint_narrative    11119301
company_public_response          6813137
company                                0
state                              60913
zip_code                           30237
tags                            14145422
consumer_consent_provided        2333406
submitted_via                          0
date_sent_to_company                   0
company_response_to_consumer          21
timely_response                        0
consumer_disputed               14121659
complaint_id                           0
dtype: int64

In [7]:
#filling missing values
df['state'] = df['state'].fillna('Unknown')
df['sub_product'] = df['sub_product'].fillna('Not specified')
df['sub_issue'] = df['sub_issue'].fillna('Not specified')


In [8]:
df = df.dropna(subset=['issue'])

In [9]:
df.isnull().sum()

date_received                          0
product                                0
sub_product                            0
issue                                  0
sub_issue                              0
consumer_complaint_narrative    11119295
company_public_response          6813136
company                                0
state                                  0
zip_code                           30237
tags                            14145417
consumer_consent_provided        2333401
submitted_via                          0
date_sent_to_company                   0
company_response_to_consumer          21
timely_response                        0
consumer_disputed               14121653
complaint_id                           0
dtype: int64

I wouldn't drop Consumer complaint narrative and Company public response because we are working on behavioural dataset we are going to get a new feature from it for our analysis

Features Engineering 

In [10]:
df['has_narrative'] = df['consumer_complaint_narrative'].notna().astype(int)
df['has_public_response'] = df['company_public_response'].notna().astype(int)

In [11]:
#Creating a binary variable for timely response
df['timely_response'] = df['timely_response'].astype(str).str.strip().str.upper()

df['timely_response_flag'] = df['timely_response'].map({
    'YES': 1,
    'NO': 0
})

In [13]:
# Creating a new variable for product category based on the 'product' column
df['product'] = df['product'].astype(str).str.lower()

df.loc[df['product'].str.contains('credit'), 'product_category'] = 'Credit'
df.loc[df['product'].str.contains('mortgage|loan|student|payday'), 'product_category'] = 'Loan'
df.loc[df['product'].str.contains('debt'), 'product_category'] = 'Debt'
df.loc[df['product'].str.contains('bank|checking|savings'), 'product_category'] = 'Banking'

df['product_category'] = df['product_category'].fillna('Other')

In [14]:
# Let convert the date columns to datetime format
df['date_received'] = pd.to_datetime(df['date_received'], errors='coerce')
df['date_sent_to_company'] = pd.to_datetime(df['date_sent_to_company'], errors='coerce')

In [ ]:
#Drop Unnecessary Columns
df = df.drop(columns=[
    'zip_code',
    'consumer_disputed',
    'tags'
], errors='ignore')

In [16]:
#checking for duplicates
df['complaint_id'].duplicated().sum()


np.int64(0)

Rechecking the dataset and making sure that they data is clean

In [20]:
df.info()

<class 'pandas.DataFrame'>
Index: 14889543 entries, 0 to 14889548
Data columns (total 19 columns):
 #   Column                        Dtype         
---  ------                        -----         
 0   date_received                 datetime64[us]
 1   product                       str           
 2   sub_product                   str           
 3   issue                         str           
 4   sub_issue                     str           
 5   consumer_complaint_narrative  str           
 6   company_public_response       str           
 7   company                       str           
 8   state                         str           
 9   consumer_consent_provided     str           
 10  submitted_via                 str           
 11  date_sent_to_company          datetime64[us]
 12  company_response_to_consumer  str           
 13  timely_response               str           
 14  complaint_id                  int64         
 15  has_narrative                 int64         
 

In [21]:
df.isnull().sum()


date_received                          0
product                                0
sub_product                            0
issue                                  0
sub_issue                              0
consumer_complaint_narrative    11119295
company_public_response          6813136
company                                0
state                                  0
consumer_consent_provided        2333401
submitted_via                          0
date_sent_to_company                   0
company_response_to_consumer          21
timely_response                        0
complaint_id                           0
has_narrative                          0
has_public_response                    0
timely_response_flag                   0
product_category                       0
dtype: int64

In [22]:
df['product_category'].value_counts()

product_category
Credit     12388357
Debt        1086030
Loan         764201
Banking      447409
Other        203546
Name: count, dtype: int64

In [23]:
df.shape

(14889543, 19)

In [ ]:
#Exporting the cleaned dataset
df.to_csv("cfpb_cleaned_data.csv", index=False)